In [1]:
# -----------------------------------------------
# IMPORTS & SETUP
# -----------------------------------------------
!pip install faker # Ensure faker is installed
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker('en_GB')
random.seed(42)
np.random.seed(42)

NUM_TRANSACTIONS = 500
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=30)

print("Libraries loaded successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.2 MB/s eta 0:00:00
Libraries loaded successfully


In [2]:
# -----------------------------------------------
# MOUNT GOOGLE DRIVE
# -----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# Set your project folder path
PROJECT_FOLDER = "/content/drive/MyDrive/compliance_ai_project"

# Create the folder if it doesn't exist
import os
os.makedirs(PROJECT_FOLDER, exist_ok=True)

print(f"Google Drive mounted successfully")
print(f"Project folder: {PROJECT_FOLDER}")

Mounted at /content/drive
Google Drive mounted successfully
Project folder: /content/drive/MyDrive/compliance_ai_project


In [3]:
# -----------------------------------------------
# REFERENCE DATA
# -----------------------------------------------
TRANSACTION_TYPES = [
    "Bank Transfer", "Mobile Wallet", "Cash Deposit",
    "Card Payment", "International Wire"
]

CHANNELS = ["Mobile App", "Online Banking", "Branch", "ATM"]

STATUSES = ["Completed", "Pending", "Failed"]
STATUS_WEIGHTS = [0.85, 0.10, 0.05]

CUSTOMER_TYPES = ["Individual", "Business"]

KYC_RISK_RATINGS = ["Low", "Medium", "High"]
KYC_WEIGHTS = [0.60, 0.30, 0.10]

NATIONALITIES = [
    "British", "British", "British", "British",
    "Nigerian", "Ghanaian", "Indian", "Pakistani",
    "Chinese", "Romanian", "Polish", "Somali",
    "Iranian", "Afghan", "Russian", "Turkish"
]

print("Reference data defined successfully")

Reference data defined successfully


In [4]:
# -----------------------------------------------
# HELPER FUNCTIONS
# -----------------------------------------------
def generate_account_number():
    return str(random.randint(10000000, 99999999))

def generate_phone_number():
    return f"07{random.randint(100000000, 999999999)}"

def generate_passport_number():
    letters = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=2))
    digits = ''.join(random.choices('0123456789', k=7))
    return letters + digits

def random_timestamp(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def generate_normal_amount(kyc_risk):
    if kyc_risk == "Low":
        return round(random.uniform(10, 2000), 2)
    elif kyc_risk == "Medium":
        return round(random.uniform(50, 8000), 2)
    else:
        return round(random.uniform(100, 20000), 2)

print("Helper functions defined successfully")

Helper functions defined successfully


In [5]:
# -----------------------------------------------
# SENDER POOL
# -----------------------------------------------
NUM_SENDERS = 50
senders = []

for i in range(NUM_SENDERS):
    kyc_risk = random.choices(KYC_RISK_RATINGS, weights=KYC_WEIGHTS)[0]
    customer_type = random.choices(CUSTOMER_TYPES, weights=[0.75, 0.25])[0]
    nationality = random.choice(NATIONALITIES)
    passport = generate_passport_number() if random.random() > 0.15 else None

    senders.append({
        "sender_account_no": generate_account_number(),
        "sender_name": fake.company() if customer_type == "Business" else fake.name(),
        "sender_phone": generate_phone_number(),
        "sender_nationality": nationality,
        "sender_passport_no": passport,
        "sender_customer_type": customer_type,
        "sender_kyc_risk": kyc_risk
    })

sender_df = pd.DataFrame(senders)
print(f"Sender pool created: {len(sender_df)} customers")
print(sender_df["sender_kyc_risk"].value_counts().to_string())

Sender pool created: 50 customers
sender_kyc_risk
Low       25
Medium    19
High       6


In [6]:
# -----------------------------------------------
# GENERATE TRANSACTIONS
# -----------------------------------------------
transactions = []

for i in range(NUM_TRANSACTIONS):
    sender = sender_df.sample(1).iloc[0]
    tx_type = random.choice(TRANSACTION_TYPES)
    timestamp = random_timestamp(START_DATE, END_DATE)
    amount = generate_normal_amount(sender["sender_kyc_risk"])

    receiver_phone = generate_phone_number()
    if tx_type == "Mobile Wallet":
        receiver_account = receiver_phone
    else:
        receiver_account = generate_account_number()

    transaction = {
        "transaction_id": f"TXN{str(i+1).zfill(5)}",
        "date_time": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "transaction_type": tx_type,
        "amount_gbp": amount,
        "status": random.choices(STATUSES, weights=STATUS_WEIGHTS)[0],
        "channel": random.choice(CHANNELS),
        "sender_account_no": sender["sender_account_no"],
        "sender_name": sender["sender_name"],
        "sender_phone": sender["sender_phone"],
        "sender_nationality": sender["sender_nationality"],
        "sender_passport_no": sender["sender_passport_no"],
        "sender_customer_type": sender["sender_customer_type"],
        "sender_kyc_risk": sender["sender_kyc_risk"],
        "receiver_account_no": receiver_account,
        "receiver_name": fake.name(),
        "receiver_phone": receiver_phone,
        "is_anomaly": False,
        "anomaly_type": None
    }
    transactions.append(transaction)

df = pd.DataFrame(transactions)
df = df.sort_values("date_time").reset_index(drop=True)
print(f"Transactions generated: {len(df)}")

Transactions generated: 500


In [7]:
# -----------------------------------------------
# PLANT ANOMALIES
# -----------------------------------------------

# Anomaly 1: Unusually Large Transactions
large_tx_indices = random.sample(range(len(df)), 10)
for idx in large_tx_indices:
    df.at[idx, "amount_gbp"] = round(random.uniform(45000, 100000), 2)
    df.at[idx, "is_anomaly"] = True
    df.at[idx, "anomaly_type"] = "Unusually Large Transaction"

# Anomaly 2: Rapid Succession / Structuring
rapid_sender = sender_df.sample(1).iloc[0]
base_time = random_timestamp(START_DATE, END_DATE)

for j in range(5):
    rapid_tx = {
        "transaction_id": f"TXN{str(NUM_TRANSACTIONS + j + 1).zfill(5)}",
        "date_time": (base_time + timedelta(minutes=j*2)).strftime("%Y-%m-%d %H:%M:%S"),
        "transaction_type": "Bank Transfer",
        "amount_gbp": round(random.uniform(1800, 2500), 2),
        "status": "Completed",
        "channel": "Online Banking",
        "sender_account_no": rapid_sender["sender_account_no"],
        "sender_name": rapid_sender["sender_name"],
        "sender_phone": rapid_sender["sender_phone"],
        "sender_nationality": rapid_sender["sender_nationality"],
        "sender_passport_no": rapid_sender["sender_passport_no"],
        "sender_customer_type": rapid_sender["sender_customer_type"],
        "sender_kyc_risk": rapid_sender["sender_kyc_risk"],
        "receiver_account_no": generate_account_number(),
        "receiver_name": fake.name(),
        "receiver_phone": generate_phone_number(),
        "is_anomaly": True,
        "anomaly_type": "Rapid Succession / Structuring"
    }
    df = pd.concat([df, pd.DataFrame([rapid_tx])], ignore_index=True)

# Anomaly 3: High Risk Customer - Large International Wire
high_risk_senders = df[df["sender_kyc_risk"] == "High"].index.tolist()
if high_risk_senders:
    hr_idx = random.choice(high_risk_senders)
    df.at[hr_idx, "transaction_type"] = "International Wire"
    df.at[hr_idx, "amount_gbp"] = round(random.uniform(30000, 75000), 2)
    df.at[hr_idx, "is_anomaly"] = True
    df.at[hr_idx, "anomaly_type"] = "High Risk Customer - Large International Wire"

# Anomaly 4: Large Transaction with Missing KYC
missing_kyc = df[df["sender_passport_no"].isna()].index.tolist()
if missing_kyc:
    mkc_idx = random.choice(missing_kyc)
    df.at[mkc_idx, "amount_gbp"] = round(random.uniform(15000, 40000), 2)
    df.at[mkc_idx, "is_anomaly"] = True
    df.at[mkc_idx, "anomaly_type"] = "Large Transaction - Missing KYC Passport"

df = df.sort_values("date_time").reset_index(drop=True)
print(f"Anomalies planted: {df['is_anomaly'].sum()}")
print(df[df['is_anomaly'] == True]['anomaly_type'].value_counts().to_string())

Anomalies planted: 17
anomaly_type
Unusually Large Transaction                      10
Rapid Succession / Structuring                    5
Large Transaction - Missing KYC Passport          1
High Risk Customer - Large International Wire     1


In [8]:
# -----------------------------------------------
# PREVIEW & EXPORT
# -----------------------------------------------

# Preview the first few rows
print("Dataset preview:")
display(df.head(10))

# Save full version
df.to_csv(f"{PROJECT_FOLDER}/transactions_full.csv", index=False)

# Save clean agent version
agent_view = df.drop(columns=["is_anomaly", "anomaly_type"], errors="ignore")
agent_view.to_csv(f"{PROJECT_FOLDER}/transactions_agent_view.csv", index=False)

print("Files saved to Google Drive successfully")
print(f"Total rows: {len(df)}")

Dataset preview:


,transaction_id,date_time,transaction_type,amount_gbp,status,channel,sender_account_no,sender_name,sender_phone,sender_nationality,sender_passport_no,sender_customer_type,sender_kyc_risk,receiver_account_no,receiver_name,receiver_phone,is_anomaly,anomaly_type
0,TXN00325,2026-03-02 17:25:16,Bank Transfer,434.71,Completed,ATM,16789850,Mrs Olivia Barnett,07800235246,British,EL4229485,Individual,Low,27810664,Mr Ian Carter,07183686071,False,None
1,TXN00018,2026-03-02 17:29:47,International Wire,5278.40,Completed,Online Banking,24305904,Anderson-Green,07935822550,Russian,QF6426077,Business,Medium,72005415,Dr Joanne Bates,07682605435,False,None
2,TXN00006,2026-03-02 19:16:22,Mobile Wallet,822.95,Completed,Branch,38716269,Dr Kenneth Marsh,07168965204,Turkish,HC2235126,Individual,Low,07840206617,Damien Walters,07840206617,False,None
3,TXN00064,2026-03-02 19:27:33,International Wire,561.78,Completed,Online Banking,74988034,Antony Sykes,07396889215,Polish,LH2737961,Individual,Low,27447248,Cheryl Burton,07255357534,False,None
4,TXN00217,2026-03-02 19:40:07,Cash Deposit,1494.15,Completed,Branch,74988034,Antony Sykes,07396889215,Polish,LH2737961,Individual,Low,19801857,Guy Brown,07720956992,False,None
5,TXN00456,2026-03-02 21:35:43,Card Payment,850.73,Failed,Branch,50477742,Jay Graham-Rose,07834717549,Pakistani,None,Individual,High,60076822,Cheryl Spencer,07314119421,False,None
6,TXN00357,2026-03-02 23:35:10,Mobile Wallet,18654.71,Completed,Branch,71070189,Thomas Group,07406284028,British,WE4240392,Business,High,07125673445,Derek Reynolds,07125673445,False,None
7,TXN00248,2026-03-02 23:35:35,Bank Transfer,830.66,Completed,Online Banking,44788100,"Forster, Turner and Fry",07907214179,Polish,XP7585874,Business,Low,41062464,Dr Colin Payne,07777210736,False,None
8,TXN00326,2026-03-03 01:32:56,Bank Transfer,5580.00,Failed,ATM,22448136,Dr Brett Jones,07507943839,Afghan,PV0863193,Individual,Medium,26185823,Julian Hobbs,07977364141,False,None
9,TXN00045,2026-03-03 01:35:25,Cash Deposit,1136.20,Pending,ATM,70505620,Gareth Robinson-Roberts,07361846370,Chinese,XU4853292,Individual,Low,16582607,Eric Ellis,07836371599,False,None


Files saved to Google Drive successfully
Total rows: 505


# Agentic Transaction Monitoring

In [9]:
!pip install anthropic


#**Step 3 — The Agent Code**

#Once that installs, add the following cells in order. I'd suggest starting a clear section in your notebook with a text cell first. To add a text cell click **+ Text** and write:

## Stage 2 — Agentic Transaction Monitoring
#This section builds the AI agent that autonomously analyses
#transactions, flags suspicious activity, and investigates
#each case with reasoned explanations.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 56.8 kB/s eta 0:00:00


In [10]:
# -----------------------------------------------
# IMPORTS & API SETUP
# -----------------------------------------------
import anthropic
import pandas as pd
import json
from google.colab import userdata

# Securely retrieve API key from Colab secrets
api_key = userdata.get('ANTHROPIC_API_KEY')

# Initialise the Anthropic client
client = anthropic.Anthropic(api_key=api_key)

print("Anthropic client initialised successfully")

Anthropic client initialised successfully


In [11]:
# -----------------------------------------------
# LOAD DATA
# -----------------------------------------------
# Load the clean file — the same view the agent sees
# No anomaly labels, just raw transaction data
df = pd.read_csv(f"{PROJECT_FOLDER}/transactions_agent_view.csv")

print(f"Transactions loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head(5))

Transactions loaded: 505
Columns: ['transaction_id', 'date_time', 'transaction_type', 'amount_gbp', 'status', 'channel', 'sender_account_no', 'sender_name', 'sender_phone', 'sender_nationality', 'sender_passport_no', 'sender_customer_type', 'sender_kyc_risk', 'receiver_account_no', 'receiver_name', 'receiver_phone']


,transaction_id,date_time,transaction_type,amount_gbp,status,channel,sender_account_no,sender_name,sender_phone,sender_nationality,sender_passport_no,sender_customer_type,sender_kyc_risk,receiver_account_no,receiver_name,receiver_phone
0,TXN00325,2026-03-02 17:25:16,Bank Transfer,434.71,Completed,ATM,16789850,Mrs Olivia Barnett,7800235246,British,EL4229485,Individual,Low,27810664,Mr Ian Carter,7183686071
1,TXN00018,2026-03-02 17:29:47,International Wire,5278.40,Completed,Online Banking,24305904,Anderson-Green,7935822550,Russian,QF6426077,Business,Medium,72005415,Dr Joanne Bates,7682605435
2,TXN00006,2026-03-02 19:16:22,Mobile Wallet,822.95,Completed,Branch,38716269,Dr Kenneth Marsh,7168965204,Turkish,HC2235126,Individual,Low,7840206617,Damien Walters,7840206617
3,TXN00064,2026-03-02 19:27:33,International Wire,561.78,Completed,Online Banking,74988034,Antony Sykes,7396889215,Polish,LH2737961,Individual,Low,27447248,Cheryl Burton,7255357534
4,TXN00217,2026-03-02 19:40:07,Cash Deposit,1494.15,Completed,Branch,74988034,Antony Sykes,7396889215,Polish,LH2737961,Individual,Low,19801857,Guy Brown,7720956992


In [12]:
# -----------------------------------------------
# STEP 1: RULE BASED FLAGGING
# -----------------------------------------------
# Rules are applied in order of severity.
# Failed transactions are handled separately with
# higher thresholds to avoid noise at scale.

flagged = []

# Pre-calculate failed transaction counts per sender
# This lets us efficiently check repeat failures
# without looping through the full dataset repeatedly
failed_counts = (
    df[df["status"] == "Failed"]
    .groupby("sender_account_no")
    .size()
    .to_dict()
)

for idx, row in df.iterrows():

    reasons = []
    is_failed = row["status"] == "Failed"

    # ---- RULES FOR COMPLETED/PENDING TRANSACTIONS ----

    if not is_failed:

        # Rule 1: Unusually large completed transaction
        if row["amount_gbp"] > 10000:
            reasons.append(
                f"Large transaction amount: £{row['amount_gbp']:,.2f}"
            )

        # Rule 2: High risk customer
        if row["sender_kyc_risk"] == "High":
            reasons.append(
                "Sender is High KYC risk (EDD required)"
            )

        # Rule 3: Missing passport on transaction over £5,000
        if pd.isna(row["sender_passport_no"]) and row["amount_gbp"] > 5000:
            reasons.append(
                "Missing KYC passport on significant transaction"
            )

        # Rule 4: International wire over £8,000
        if (row["transaction_type"] == "International Wire"
                and row["amount_gbp"] > 8000):
            reasons.append(
                f"Large international wire: £{row['amount_gbp']:,.2f}"
            )

    # ---- RULES FOR FAILED TRANSACTIONS ----
    # Higher thresholds applied deliberately to reduce
    # noise — only genuinely suspicious failures flagged

    else:

        # Rule 5: Large failed transaction
        # A failed attempt at this value warrants scrutiny
        # regardless of other factors — intent is the concern
        if row["amount_gbp"] > 15000:
            reasons.append(
                f"Large failed transaction attempt: "
                f"£{row['amount_gbp']:,.2f} — "
                f"suspicious intent cannot be ruled out"
            )

        # Rule 6: Repeat failed transactions from same sender
        # 3+ failures in the period suggests account probing
        # or repeated attempts to bypass controls
        sender_failed_count = failed_counts.get(
            row["sender_account_no"], 0
        )
        if sender_failed_count >= 3:
            reasons.append(
                f"Repeat failed transactions: sender has "
                f"{sender_failed_count} failed transactions "
                f"in the review period — possible control probing"
            )

        # Rule 7: High risk customer with any failed transaction
        # For EDD customers, even a single failed transaction
        # warrants documentation and review
        if row["sender_kyc_risk"] == "High":
            reasons.append(
                "High KYC risk sender — failed transaction "
                "requires documentation under EDD obligations"
            )

    # If any rules triggered, add to flagged list
    if reasons:
        flagged.append({
            "transaction_id": row["transaction_id"],
            "date_time": row["date_time"],
            "transaction_type": row["transaction_type"],
            "amount_gbp": row["amount_gbp"],
            "status": row["status"],
            "channel": row["channel"],
            "sender_account_no": row["sender_account_no"],
            "sender_name": row["sender_name"],
            "sender_nationality": row["sender_nationality"],
            "sender_passport_no": row["sender_passport_no"],
            "sender_customer_type": row["sender_customer_type"],
            "sender_kyc_risk": row["sender_kyc_risk"],
            "receiver_account_no": row["receiver_account_no"],
            "receiver_name": row["receiver_name"],
            "flag_reasons": reasons
        })

print(f"Completed/Pending transactions flagged : "
      f"{len([f for f in flagged if f['status'] != 'Failed'])}")
print(f"Failed transactions flagged            : "
      f"{len([f for f in flagged if f['status'] == 'Failed'])}")
print(f"Total flagged                          : {len(flagged)}")

Completed/Pending transactions flagged : 60
Failed transactions flagged            : 12
Total flagged                          : 72


In [13]:
# -----------------------------------------------
# STEP 2: RAPID SUCCESSION DETECTION
# -----------------------------------------------
# This rule needs separate treatment because it
# looks across multiple rows for the same sender
# rather than analysing one transaction at a time

df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

# Group transactions by sender account
for account_no, group in df.groupby("sender_account_no"):
    group = group.sort_values("date_time").reset_index(drop=True)

    # Slide a window looking for 3+ transactions within 30 minutes
    for i in range(len(group)):
        window = group[
            (group["date_time"] >= group.at[i, "date_time"]) &
            (group["date_time"] <= group.at[i, "date_time"] + pd.Timedelta(minutes=30))
        ]

        if len(window) >= 3:
            # Check if these transactions are already flagged
            for _, tx in window.iterrows():
                already_flagged = any(
                    f["transaction_id"] == tx["transaction_id"] for f in flagged
                )
                if not already_flagged:
                    flagged.append({
                        "transaction_id": tx["transaction_id"],
                        "date_time": str(tx["date_time"]),
                        "transaction_type": tx["transaction_type"],
                        "amount_gbp": tx["amount_gbp"],
                        "status": tx["status"],
                        "channel": tx["channel"],
                        "sender_account_no": tx["sender_account_no"],
                        "sender_name": tx["sender_name"],
                        "sender_nationality": tx["sender_nationality"],
                        "sender_passport_no": tx["sender_passport_no"],
                        "sender_customer_type": tx["sender_customer_type"],
                        "sender_kyc_risk": tx["sender_kyc_risk"],
                        "receiver_account_no": tx["receiver_account_no"],
                        "receiver_name": tx["receiver_name"],
                        "flag_reasons": ["Rapid succession transactions - potential structuring"]
                    })
            break  # Avoid duplicate window detection for same sender

print(f"Total flagged after rapid succession check: {len(flagged)}")

Total flagged after rapid succession check: 77


In [14]:
# -----------------------------------------------
# STEP 3: AI INVESTIGATION LOOP
# -----------------------------------------------
import time
import re
import threading

investigated = []
TIMEOUT_SECONDS = 15  # Maximum wait per API call

def call_claude_api(client, prompt, result_container):
    """
    Runs the API call in a separate thread.
    Stores result in result_container so the main
    thread can retrieve it after the timeout.
    """
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            messages=[{"role": "user", "content": prompt}]
        )
        result_container["assessment"] = response.content[0].text
        result_container["success"] = True
    except Exception as e:
        result_container["error"] = str(e)
        result_container["success"] = False


def extract_risk_score(assessment):
    """Extracts risk score from varied Claude response formats."""
    risk_score = "Unknown"
    lines = assessment.split("\n")
    for j, line in enumerate(lines):
        clean_line = re.sub(r'[#*]', '', line).strip()
        if "RISK SCORE:" in clean_line:
            score_text = clean_line.split("RISK SCORE:")[-1].strip()
            if score_text:
                for level in ["Critical", "High", "Medium", "Low"]:
                    if level.lower() in score_text.lower():
                        return level
            else:
                # Check next line
                if j + 1 < len(lines):
                    next_line = re.sub(r'[#*]', '', lines[j+1]).strip()
                    for level in ["Critical", "High", "Medium", "Low"]:
                        if level.lower() in next_line.lower():
                            return level
    return risk_score


for i, tx in enumerate(flagged):
    print(
        f"Investigating {i+1}/{len(flagged)}: "
        f"{tx['transaction_id']}...", end=" "
    )

    # Build the prompt
    prompt = f"""
You are a senior compliance analyst at a UK bank reviewing flagged transactions.
Analyse the following transaction and provide a structured assessment.

TRANSACTION DETAILS:
- Transaction ID: {tx['transaction_id']}
- Date & Time: {tx['date_time']}
- Type: {tx['transaction_type']}
- Amount: £{tx['amount_gbp']:,.2f}
- Status: {tx['status']}
- Channel: {tx['channel']}

SENDER DETAILS:
- Name: {tx['sender_name']}
- Account: {tx['sender_account_no']}
- Nationality: {tx['sender_nationality']}
- Passport: {tx['sender_passport_no'] if pd.notna(tx['sender_passport_no']) else 'MISSING - KYC INCOMPLETE'}
- Customer Type: {tx['sender_customer_type']}
- KYC Risk Rating: {tx['sender_kyc_risk']}

RECEIVER DETAILS:
- Name: {tx['receiver_name']}
- Account: {tx['receiver_account_no']}

FLAGS TRIGGERED:
{chr(10).join(f'- {r}' for r in tx['flag_reasons'])}

Provide your assessment in this exact format:
RISK SCORE: [Low / Medium / High / Critical]
SUMMARY: [2-3 sentence summary of why this transaction is suspicious]
RECOMMENDED ACTION: [What the compliance team should do next]
"""

    # Run API call in a separate thread with timeout
    result_container = {"success": False, "assessment": None, "error": None}
    thread = threading.Thread(
        target=call_claude_api,
        args=(client, prompt, result_container)
    )
    thread.start()
    thread.join(timeout=TIMEOUT_SECONDS)

    # Check what happened
    if thread.is_alive():
        # Thread is still running — API call timed out
        print(f"Timeout — skipping")
        assessment = "Assessment timed out — manual review required"
        risk_score = "Unknown"

    elif result_container["success"]:
        # API call succeeded
        assessment = result_container["assessment"]
        risk_score = extract_risk_score(assessment)
        print(f"Done — {risk_score}")

    else:
        # API call failed with an error
        print(f"Error — {result_container['error']}")
        assessment = f"Assessment failed: {result_container['error']}"
        risk_score = "Unknown"

    # Store result
    investigated.append({
        **tx,
        "ai_risk_score": risk_score,
        "ai_assessment": assessment
    })

    # Small delay between calls
    time.sleep(0.5)

# Summary
timeout_count = len([
    r for r in investigated
    if "timed out" in r["ai_assessment"]
])
success_count = len(investigated) - timeout_count

print(f"\nInvestigation complete.")
print(f"Successfully assessed : {success_count}")
print(f"Timed out (skipped)   : {timeout_count}")
print(f"Total processed       : {len(investigated)}")

Investigating 1/77: TXN00456... Done — Critical
Investigating 2/77: TXN00357... Done — High
Investigating 3/77: TXN00350... Done — High
Investigating 4/77: TXN00389... Done — Medium
Investigating 5/77: TXN00294... Done — High
Investigating 6/77: TXN00225... Done — High
Investigating 7/77: TXN00459... Done — High
Investigating 8/77: TXN00303... Done — Critical
Investigating 9/77: TXN00131... Done — Medium
Investigating 10/77: TXN00354... Done — High
Investigating 11/77: TXN00108... Done — High
Investigating 12/77: TXN00256... Done — High
Investigating 13/77: TXN00083... Done — High
Investigating 14/77: TXN00255... Done — Medium
Investigating 15/77: TXN00408... Done — Medium
Investigating 16/77: TXN00058... Done — Critical
Investigating 17/77: TXN00495... Done — Medium
Investigating 18/77: TXN00291... Done — High
Investigating 19/77: TXN00447... Done — High
Investigating 20/77: TXN00186... Done — Medium
Investigating 21/77: TXN00477... Done — Medium
Investigating 22/77: TXN00392... Done 

In [15]:
# -----------------------------------------------
# STEP 4: REVIEW & SAVE RESULTS
# -----------------------------------------------

# Convert to dataframe for easy viewing
results_df = pd.DataFrame(investigated)

# Show summary of risk scores assigned by the agent
print("Risk Score Summary:")
print(results_df["ai_risk_score"].value_counts().to_string())
print()

# Preview a sample assessment so you can see the agent's reasoning
sample = investigated[0]
print(f"Sample Investigation — {sample['transaction_id']}:")
print("-" * 50)
print(sample["ai_assessment"])
print("-" * 50)

# Save results to CSV for Stage 3
# Save results to CSV for Stage 3
results_df.to_csv(f"{PROJECT_FOLDER}/investigation_results.csv", index=False)
print("\nResults saved to Google Drive successfully")

Risk Score Summary:
ai_risk_score
High        28
Medium      25
Critical    23
Low          1

Sample Investigation — TXN00456:
--------------------------------------------------
RISK SCORE: Critical

SUMMARY: This transaction presents critical compliance risk due to the sender's incomplete KYC status (missing passport documentation) combined with a high-risk rating and a pattern of 7 failed transactions suggesting potential control probing or account testing. The failed payment itself, while not completed, occurs in a context of elevated AML/CFT concern that requires immediate remediation before any further activity is permitted on this account.

RECOMMENDED ACTION: 
1. **Immediate**: Suspend account 50477742 pending KYC completion. Request mandatory passport documentation and full identity verification from Jay Graham-Rose within 5 business days.
2. **Enhanced Due Diligence**: Conduct EDD investigation into the source of funds, purpose of transactions, and beneficial ownership given 

In [16]:
# -----------------------------------------------
# SENSE CHECK — Agent Performance Summary
# -----------------------------------------------

# Risk score breakdown
print("=" * 50)
print("RISK SCORE BREAKDOWN")
print("=" * 50)
print(results_df["ai_risk_score"].value_counts().to_string())

# Show one example of each risk level so you can
# review the quality of the agent's reasoning
print("\n" + "=" * 50)
print("SAMPLE ASSESSMENTS BY RISK LEVEL")
print("=" * 50)

for risk_level in ["Critical", "High", "Medium", "Low"]:
    sample = results_df[results_df["ai_risk_score"] == risk_level]
    if len(sample) > 0:
        row = sample.iloc[0]
        print(f"\n--- {risk_level} Risk Example ---")
        print(f"Transaction: {row['transaction_id']}")
        print(f"Amount: £{row['amount_gbp']:,.2f}")
        print(f"Sender KYC Risk: {row['sender_kyc_risk']}")
        print(f"Assessment:\n{row['ai_assessment']}")
        print("-" * 50)

RISK SCORE BREAKDOWN
ai_risk_score
High        28
Medium      25
Critical    23
Low          1

SAMPLE ASSESSMENTS BY RISK LEVEL

--- Critical Risk Example ---
Transaction: TXN00456
Amount: £850.73
Sender KYC Risk: High
Assessment:
RISK SCORE: Critical

SUMMARY: This transaction presents critical compliance risk due to the sender's incomplete KYC status (missing passport documentation) combined with a high-risk rating and a pattern of 7 failed transactions suggesting potential control probing or account testing. The failed payment itself, while not completed, occurs in a context of elevated AML/CFT concern that requires immediate remediation before any further activity is permitted on this account.

RECOMMENDED ACTION: 
1. **Immediate**: Suspend account 50477742 pending KYC completion. Request mandatory passport documentation and full identity verification from Jay Graham-Rose within 5 business days.
2. **Enhanced Due Diligence**: Conduct EDD investigation into the source of funds, pur

In [17]:
# -----------------------------------------------
# REPAIR RISK SCORES IN EXISTING RESULTS
# -----------------------------------------------
import pandas as pd
import re

results_df = pd.read_csv(f"{PROJECT_FOLDER}/investigation_results.csv")

def extract_risk_score(assessment):
    """Extracts risk score from varied Claude response formats."""
    if pd.isna(assessment):
        return "Unknown"

    lines = assessment.split("\n")

    for i, line in enumerate(lines):
        # Strip markdown formatting
        clean_line = re.sub(r'[#*]', '', line).strip()

        if "RISK SCORE:" in clean_line:
            # Try to get score from same line
            score_text = clean_line.split("RISK SCORE:")[-1].strip()

            if score_text:
                # Check if a valid risk level is in the text
                for level in ["Critical", "High", "Medium", "Low"]:
                    if level.lower() in score_text.lower():
                        return level

            # If not found on same line, check next line
            if i + 1 < len(lines):
                next_line = re.sub(r'[#*]', '', lines[i+1]).strip()
                for level in ["Critical", "High", "Medium", "Low"]:
                    if level.lower() in next_line.lower():
                        return level

    return "Unknown"

# Apply the fix to all rows
results_df["ai_risk_score"] = results_df["ai_assessment"].apply(extract_risk_score)

# Save the repaired file
results_df.to_csv(f"{PROJECT_FOLDER}/investigation_results.csv", index=False)

# Show the updated breakdown
print("Risk Score Breakdown (Repaired):")
print("=" * 40)
print(results_df["ai_risk_score"].value_counts().to_string())
print(f"\nUnknown remaining: {(results_df['ai_risk_score'] == 'Unknown').sum()}")

Risk Score Breakdown (Repaired):
ai_risk_score
High        28
Medium      25
Critical    23
Low          1

Unknown remaining: 0


## Stage 3 — Automated Compliance Report
This section takes the agent's investigation results and
generates a professionally formatted PDF compliance report.



In [18]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.1 MB/s eta 0:00:00


In [19]:
# -----------------------------------------------
# IMPORTS & LOAD RESULTS
# -----------------------------------------------
import pandas as pd
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, PageBreak, HRFlowable
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

# Load investigation results
results_df = pd.read_csv(f"{PROJECT_FOLDER}/investigation_results.csv")

# Report generation timestamp
REPORT_DATE = datetime.now().strftime("%d %B %Y")
REPORT_TIME = datetime.now().strftime("%H:%M:%S")

print(f"Results loaded: {len(results_df)} transactions")
print(f"Report date: {REPORT_DATE}")

Results loaded: 77 transactions
Report date: 01 April 2026


In [20]:
# -----------------------------------------------
# DEFINE STYLES
# -----------------------------------------------
# These styles control fonts, sizes and colours
# throughout the report for a consistent look

styles = getSampleStyleSheet()

# Colour palette — professional navy and grey
NAVY = colors.HexColor('#1B2A4A')
DARK_GREY = colors.HexColor('#4A4A4A')
LIGHT_GREY = colors.HexColor('#F5F5F5')
MID_GREY = colors.HexColor('#CCCCCC')
WHITE = colors.white

# Risk score colours
CRITICAL_RED = colors.HexColor('#C0392B')
HIGH_ORANGE = colors.HexColor('#E67E22')
MEDIUM_YELLOW = colors.HexColor('#F39C12')
LOW_GREEN = colors.HexColor('#27AE60')

# Custom styles
style_cover_title = ParagraphStyle(
    'CoverTitle',
    parent=styles['Title'],
    fontSize=28,
    textColor=WHITE,
    alignment=TA_CENTER,
    spaceAfter=6
)

style_cover_subtitle = ParagraphStyle(
    'CoverSubtitle',
    parent=styles['Normal'],
    fontSize=14,
    textColor=WHITE,
    alignment=TA_CENTER,
    spaceAfter=4
)

style_cover_meta = ParagraphStyle(
    'CoverMeta',
    parent=styles['Normal'],
    fontSize=11,
    textColor=colors.HexColor('#BDC3C7'),
    alignment=TA_CENTER,
    spaceAfter=4
)

style_section_heading = ParagraphStyle(
    'SectionHeading',
    parent=styles['Heading1'],
    fontSize=14,
    textColor=NAVY,
    spaceBefore=12,
    spaceAfter=6,
    borderPad=4
)

style_body = ParagraphStyle(
    'Body',
    parent=styles['Normal'],
    fontSize=10,
    textColor=DARK_GREY,
    spaceAfter=6,
    leading=14
)

style_transaction_id = ParagraphStyle(
    'TransactionID',
    parent=styles['Normal'],
    fontSize=11,
    textColor=NAVY,
    fontName='Helvetica-Bold',
    spaceBefore=10,
    spaceAfter=4
)

style_assessment = ParagraphStyle(
    'Assessment',
    parent=styles['Normal'],
    fontSize=9,
    textColor=DARK_GREY,
    spaceAfter=4,
    leading=13,
    leftIndent=10
)

style_footer = ParagraphStyle(
    'Footer',
    parent=styles['Normal'],
    fontSize=8,
    textColor=MID_GREY,
    alignment=TA_CENTER
)

print("Styles defined successfully")

Styles defined successfully


In [21]:
# -----------------------------------------------
# HELPER FUNCTIONS
# -----------------------------------------------
import re

def get_risk_colour(risk_score):
    """Returns the colour associated with a risk level."""
    colours = {
        "Critical": CRITICAL_RED,
        "High": HIGH_ORANGE,
        "Medium": MEDIUM_YELLOW,
        "Low": LOW_GREEN
    }
    return colours.get(risk_score, DARK_GREY)


def clean_assessment_text(text):
    """
    Cleans markdown formatting from Claude's responses
    so they render cleanly as plain text in the PDF.
    """
    if pd.isna(text):
        return "Assessment unavailable."
    # Remove markdown headers and bold symbols
    text = re.sub(r'#+\s*', '', text)
    text = re.sub(r'\*\*', '', text)
    text = re.sub(r'\*', '', text)
    text = re.sub(r'---+', '', text)
    # Clean up excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def add_page_footer(canvas, doc):
    """Adds a confidential footer to every page."""
    canvas.saveState()
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(MID_GREY)

    # Footer line
    canvas.setStrokeColor(MID_GREY)
    canvas.line(20*mm, 15*mm, 190*mm, 15*mm)

    # Footer text
    canvas.drawString(
        20*mm, 10*mm,
        "CONFIDENTIAL — For Internal Use Only"
    )
    canvas.drawRightString(
        190*mm, 10*mm,
        f"Page {doc.page} | Generated: {REPORT_DATE}"
    )
    canvas.restoreState()


print("Helper functions defined successfully")

Helper functions defined successfully


In [22]:
# @title
# -----------------------------------------------
# BUILD REPORT CONTENT
# -----------------------------------------------

story = []  # This list holds every element of the report in order

# ==================================================
# COVER PAGE
# ==================================================

# Navy background rectangle for cover
from reportlab.platypus import FrameBreak
from reportlab.lib.units import inch

# Spacer to push content down on cover page
story.append(Spacer(1, 40*mm))

# Classification badge
badge_data = [["CONFIDENTIAL — INTERNAL USE ONLY"]]
badge_table = Table(badge_data, colWidths=[160*mm])
badge_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#C0392B')),
    ('TEXTCOLOR', (0,0), (-1,-1), WHITE),
    ('FONTNAME', (0,0), (-1,-1), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 9),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
]))
story.append(badge_table)
story.append(Spacer(1, 15*mm))

# Report title
story.append(Paragraph(
    "TRANSACTION MONITORING",
    ParagraphStyle('T1', parent=styles['Title'],
                   fontSize=26, textColor=NAVY,
                   alignment=TA_CENTER, spaceAfter=4)
))
story.append(Paragraph(
    "COMPLIANCE REPORT",
    ParagraphStyle('T2', parent=styles['Title'],
                   fontSize=26, textColor=NAVY,
                   alignment=TA_CENTER, spaceAfter=4)
))

story.append(Spacer(1, 8*mm))
story.append(HRFlowable(width="80%", thickness=2,
                         color=NAVY, spaceAfter=8*mm))

# Report metadata table
meta_data = [
    ["Report Date:", REPORT_DATE],
    ["Generated At:", REPORT_TIME],
    ["Classification:", "Confidential"],
    ["Department:", "Compliance & Financial Crime"],
    ["Review Period:", "Last 30 Days"],
    ["Total Transactions Reviewed:", f"{len(results_df)} flagged from 500"],
]

meta_table = Table(meta_data, colWidths=[60*mm, 100*mm])
meta_table.setStyle(TableStyle([
    ('FONTNAME', (0,0), (-1,-1), 'Helvetica'),
    ('FONTSIZE', (0,0), (-1,-1), 11),
    ('TEXTCOLOR', (0,0), (0,-1), NAVY),
    ('FONTNAME', (0,0), (0,-1), 'Helvetica-Bold'),
    ('TEXTCOLOR', (1,0), (1,-1), DARK_GREY),
    ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ('TOPPADDING', (0,0), (-1,-1), 5),
    ('LINEBELOW', (0,-1), (-1,-1), 0.5, MID_GREY),
]))
story.append(meta_table)

story.append(PageBreak())


# ==================================================
# EXECUTIVE SUMMARY
# ==================================================

story.append(Paragraph("1. Executive Summary", style_section_heading))
story.append(HRFlowable(width="100%", thickness=1,
                         color=NAVY, spaceAfter=6))

# Calculate key stats
total_flagged = len(results_df)
critical_count = len(results_df[results_df["ai_risk_score"] == "Critical"])
high_count = len(results_df[results_df["ai_risk_score"] == "High"])
medium_count = len(results_df[results_df["ai_risk_score"] == "Medium"])
low_count = len(results_df[results_df["ai_risk_score"] == "Low"])
missing_kyc = len(results_df[results_df["sender_passport_no"].isna()])
total_amount = results_df["amount_gbp"].sum()
critical_high_amount = results_df[
    results_df["ai_risk_score"].isin(["Critical", "High"])
]["amount_gbp"].sum()

# Count manual review cases for executive summary
manual_count = len(results_df[
    results_df["ai_assessment"].str.contains(
        "timed out|failed|unavailable",
        case=False,
        na=False
    )
])

summary_text = f"""
This report presents the findings of an automated AI-assisted transaction monitoring
review covering the most recent 30-day period. A total of 500 transactions were
processed through the monitoring system, of which <b>{total_flagged} transactions
({(total_flagged/500*100):.1f}%)</b> were flagged for further review based on
rule-based detection and AI-assisted risk assessment.
<br/><br/>
The review identified <b>{critical_count} Critical</b> and <b>{high_count} High</b>
risk transactions requiring immediate compliance action, collectively involving
<b>£{critical_high_amount:,.2f}</b> in transaction value. Additionally,
<b>{missing_kyc} transactions</b> were identified where the sender had incomplete
KYC documentation, representing a regulatory compliance gap that requires
urgent remediation.
<br/><br/>
{"<b>Important:</b> " + str(manual_count) + " transaction(s) could not be automatically assessed due to system timeouts and <b>require urgent manual review</b> by a compliance analyst before end of business. These cases are detailed in Section 4 of this report.<br/><br/>" if manual_count > 0 else ""}
All remaining flagged transactions have been assessed by the AI compliance monitoring
agent and assigned risk scores with recommended actions. This report should be
reviewed by the Compliance team and escalated as appropriate.
"""

story.append(Paragraph(summary_text, style_body))
story.append(Spacer(1, 6*mm))


# ==================================================
# RISK SCORE SUMMARY TABLE
# ==================================================

story.append(Paragraph("2. Risk Score Summary", style_section_heading))
story.append(HRFlowable(width="100%", thickness=1,
                         color=NAVY, spaceAfter=6))

# Stats cards as a table
stats_data = [
    ["Risk Level", "Transaction Count", "% of Flagged", "Action Required"],
    ["Critical", str(critical_count),
     f"{(critical_count/total_flagged*100):.1f}%",
     "Immediate escalation"],
    ["High", str(high_count),
     f"{(high_count/total_flagged*100):.1f}%",
     "Review within 24 hours"],
    ["Medium", str(medium_count),
     f"{(medium_count/total_flagged*100):.1f}%",
     "Review within 5 days"],
    ["Low", str(low_count),
     f"{(low_count/total_flagged*100):.1f}%",
     "Monitor and document"],
    ["TOTAL", str(total_flagged), "100%", ""],
]

stats_table = Table(stats_data,
                    colWidths=[40*mm, 45*mm, 40*mm, 65*mm])
stats_table.setStyle(TableStyle([
    # Header row
    ('BACKGROUND', (0,0), (-1,0), NAVY),
    ('TEXTCOLOR', (0,0), (-1,0), WHITE),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 10),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    # Risk level colours
    ('BACKGROUND', (0,1), (0,1), CRITICAL_RED),
    ('TEXTCOLOR', (0,1), (0,1), WHITE),
    ('FONTNAME', (0,1), (0,1), 'Helvetica-Bold'),
    ('BACKGROUND', (0,2), (0,2), HIGH_ORANGE),
    ('TEXTCOLOR', (0,2), (0,2), WHITE),
    ('FONTNAME', (0,2), (0,2), 'Helvetica-Bold'),
    ('BACKGROUND', (0,3), (0,3), MEDIUM_YELLOW),
    ('FONTNAME', (0,3), (0,3), 'Helvetica-Bold'),
    ('BACKGROUND', (0,4), (0,4), LOW_GREEN),
    ('TEXTCOLOR', (0,4), (0,4), WHITE),
    ('FONTNAME', (0,4), (0,4), 'Helvetica-Bold'),
    # Total row
    ('BACKGROUND', (0,5), (-1,5), LIGHT_GREY),
    ('FONTNAME', (0,5), (-1,5), 'Helvetica-Bold'),
    ('TEXTCOLOR', (0,5), (-1,5), NAVY),
    # Grid
    ('GRID', (0,0), (-1,-1), 0.5, MID_GREY),
    ('ROWBACKGROUNDS', (1,1), (-1,4), [WHITE, LIGHT_GREY]),
]))

story.append(stats_table)
story.append(PageBreak())


# ==================================================
# DETAILED FINDINGS
# ==================================================

story.append(Paragraph("3. Detailed Findings", style_section_heading))
story.append(HRFlowable(width="100%", thickness=1,
                         color=NAVY, spaceAfter=6))

intro_text = """
The following section details all flagged transactions, ordered by risk score
(Critical first). Each entry includes transaction details, the flags triggered
by the monitoring system, and the AI compliance agent's assessment and
recommended actions.
"""
story.append(Paragraph(intro_text, style_body))
story.append(Spacer(1, 4*mm))

# Order by risk severity
risk_order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
results_df["risk_order"] = results_df["ai_risk_score"].map(risk_order)
sorted_df = results_df.sort_values("risk_order").reset_index(drop=True)

# Loop through every flagged transaction
for i, row in sorted_df.iterrows():

    risk_colour = get_risk_colour(row["ai_risk_score"])

    # Transaction header bar
    header_data = [[
        f"  {row['transaction_id']}",
        f"Risk: {row['ai_risk_score']}  "
    ]]
    header_table = Table(header_data, colWidths=[130*mm, 60*mm])
    header_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), NAVY),
        ('TEXTCOLOR', (0,0), (0,0), WHITE),
        ('TEXTCOLOR', (1,0), (1,0), WHITE),
        ('FONTNAME', (0,0), (-1,-1), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 10),
        ('ALIGN', (0,0), (0,0), 'LEFT'),
        ('ALIGN', (1,0), (1,0), 'RIGHT'),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('LEFTPADDING', (0,0), (0,0), 8),
        ('RIGHTPADDING', (1,0), (1,0), 8),
    ]))
    story.append(header_table)

    # Transaction details table
    passport = row['sender_passport_no'] if pd.notna(
        row['sender_passport_no']) else "MISSING — KYC INCOMPLETE"

    details_data = [
        ["Date & Time", str(row['date_time']),
         "Type", row['transaction_type']],
        ["Amount (GBP)", f"£{float(row['amount_gbp']):,.2f}",
         "Status", row['status']],
        ["Channel", row['channel'],
         "Sender KYC Risk", row['sender_kyc_risk']],
        ["Sender Name", row['sender_name'],
         "Sender Account", row['sender_account_no']],
        ["Sender Nationality", row['sender_nationality'],
         "Passport No", passport],
        ["Receiver Name", row['receiver_name'],
         "Receiver Account", row['receiver_account_no']],
    ]

    details_table = Table(details_data,
                          colWidths=[38*mm, 57*mm, 38*mm, 57*mm])
    details_table.setStyle(TableStyle([
        ('FONTNAME', (0,0), (-1,-1), 'Helvetica'),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('FONTNAME', (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTNAME', (2,0), (2,-1), 'Helvetica-Bold'),
        ('TEXTCOLOR', (0,0), (0,-1), NAVY),
        ('TEXTCOLOR', (2,0), (2,-1), NAVY),
        ('BACKGROUND', (0,0), (-1,-1), LIGHT_GREY),
        ('ROWBACKGROUNDS', (0,0), (-1,-1), [WHITE, LIGHT_GREY]),
        ('GRID', (0,0), (-1,-1), 0.3, MID_GREY),
        ('TOPPADDING', (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ('LEFTPADDING', (0,0), (-1,-1), 6),
    ]))
    story.append(details_table)

    # AI Assessment
    clean_text = clean_assessment_text(row['ai_assessment'])
    story.append(Paragraph(
        "<b>AI Compliance Assessment:</b>",
        ParagraphStyle('AssessHead', parent=style_body,
                       textColor=NAVY, spaceBefore=4)
    ))
    story.append(Paragraph(clean_text, style_assessment))
    story.append(Spacer(1, 4*mm))


story.append(PageBreak())
# ==================================================
# MANUAL REVIEW SECTION
# ==================================================

# Filter transactions needing manual review
manual_df = results_df[
    results_df["ai_assessment"].str.contains(
        "timed out|failed|unavailable",
        case=False,
        na=False
    )
].copy()

story.append(Paragraph(
    f"4. Manual Review Required ({len(manual_df)})",
    style_section_heading
))
story.append(HRFlowable(
    width="100%", thickness=1,
    color=HIGH_ORANGE, spaceAfter=6
))

if len(manual_df) == 0:
    story.append(Paragraph(
        "All flagged transactions were successfully assessed by the AI "
        "monitoring system. No manual reviews outstanding.",
        style_body
    ))
else:
    story.append(Paragraph(
        f"The following {len(manual_df)} transaction(s) could not be assessed "
        "automatically by the AI monitoring system due to a timeout or API "
        "error during processing. These cases must be reviewed manually by a "
        "compliance analyst before end of business. They should be treated as "
        "high priority until a risk assessment has been completed and documented.",
        style_body
    ))
    story.append(Spacer(1, 4*mm))

    # Manual review table
    manual_data = [
        [
            Paragraph("<b>Transaction ID</b>", style_assessment),
            Paragraph("<b>Date & Time</b>", style_assessment),
            Paragraph("<b>Amount (GBP)</b>", style_assessment),
            Paragraph("<b>Type</b>", style_assessment),
            Paragraph("<b>Sender Name</b>", style_assessment),
            Paragraph("<b>KYC Risk</b>", style_assessment),
            Paragraph("<b>Reason</b>", style_assessment),
        ]
    ]
    for i, row in manual_df.iterrows():
        manual_data.append([
            Paragraph(str(row['transaction_id']), style_assessment),
            Paragraph(str(row['date_time']), style_assessment),
            Paragraph(f"£{float(row['amount_gbp']):,.2f}", style_assessment),
            Paragraph(str(row['transaction_type']), style_assessment),
            Paragraph(str(row['sender_name']), style_assessment),
            Paragraph(str(row['sender_kyc_risk']), style_assessment),
            Paragraph("AI timeout — manual review needed", style_assessment),
        ])

    manual_table = Table(
        manual_data,
        colWidths=[22*mm, 30*mm, 22*mm, 25*mm, 38*mm, 16*mm, 37*mm]
    )
    manual_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), HIGH_ORANGE),
        ('TEXTCOLOR',  (0,0), (-1,0), WHITE),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE',   (0,0), (-1,-1), 7),
        ('ALIGN',      (0,0), (-1,-1), 'LEFT'),
        ('VALIGN',     (0,0), (-1,-1), 'TOP'),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [WHITE, LIGHT_GREY]),
        ('GRID',       (0,0), (-1,-1), 0.3, MID_GREY),
        ('TOPPADDING',    (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ('LEFTPADDING',   (0,0), (-1,-1), 3),
    ]))
    story.append(manual_table)
    story.append(Spacer(1, 4*mm))

    # Instruction box
    instruction_data = [[
        Paragraph(
            "<b>ACTION REQUIRED:</b> Each transaction listed above must be "
            "manually reviewed and assigned a risk score by a compliance analyst. "
            "Once assessed, update the case management system with findings and "
            "recommended actions. Do not close these cases until assessment "
            "is documented.",
            ParagraphStyle('ManualInst', parent=styles['Normal'],
                           fontSize=9, textColor=DARK_GREY, leading=13)
        )
    ]]
    instruction_box = Table(instruction_data, colWidths=[170*mm])
    instruction_box.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,-1), colors.HexColor('#FEF9E7')),
        ('TOPPADDING',    (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LEFTPADDING',   (0,0), (-1,-1), 10),
        ('RIGHTPADDING',  (0,0), (-1,-1), 10),
        ('BOX', (0,0), (-1,-1), 1, HIGH_ORANGE),
    ]))
    story.append(instruction_box)

story.append(PageBreak())


# ==================================================
# RECOMMENDATIONS
# ==================================================

story.append(Paragraph("5. Recommendations", style_section_heading))
story.append(HRFlowable(width="100%", thickness=1,
                         color=NAVY, spaceAfter=6))

recommendations = [
    ("<b>Immediate Escalation (Critical Transactions):</b>",
     f"All {critical_count} Critical risk transactions should be escalated to the "
     f"Senior Compliance Officer immediately. Where transactions are still pending, "
     f"consider placing a temporary hold pending investigation."),

    ("<b>24-Hour Review (High Risk Transactions):</b>",
     f"The {high_count} High risk transactions should be assigned to compliance "
     f"analysts for review within 24 hours. Each case should be documented with "
     f"outcome notes in the case management system."),

    ("<b>KYC Remediation:</b>",
     f"{missing_kyc} transactions were identified where the sender has incomplete "
     f"KYC documentation (missing passport). The KYC/Onboarding team should "
     f"contact affected customers within 10 business days to obtain outstanding "
     f"documentation. Transaction restrictions should be considered until "
     f"remediated."),

    ("<b>Structuring Review:</b>",
     "Rapid succession transactions flagged in this report should be reviewed "
     "for potential structuring activity. If structuring is confirmed, a "
     "Suspicious Activity Report (SAR) should be filed with the National Crime "
     "Agency (NCA) within the required timeframe."),

    ("<b>Enhanced Due Diligence:</b>",
     "All High KYC risk customers involved in flagged transactions should have "
     "their EDD documentation reviewed and confirmed as current. Any gaps should "
     "be escalated to the Relationship Manager."),
]

for heading, body in recommendations:
    story.append(Paragraph(heading, style_body))
    story.append(Paragraph(body, ParagraphStyle(
        'RecBody', parent=style_body, leftIndent=10
    )))
    story.append(Spacer(1, 3*mm))

story.append(Spacer(1, 8*mm))
story.append(HRFlowable(width="100%", thickness=1,
                         color=MID_GREY, spaceAfter=4))
story.append(Paragraph(
    "This report was generated automatically by the AI Transaction Monitoring "
    "System. All findings should be reviewed and validated by a qualified "
    "compliance professional before action is taken.",
    ParagraphStyle('Disclaimer', parent=style_body,
                   fontSize=8, textColor=MID_GREY)
))

print("Report content built successfully")
print(f"Total story elements: {len(story)}")

Report content built successfully
Total story elements: 435


In [23]:
# -----------------------------------------------
# GENERATE PDF
# -----------------------------------------------

OUTPUT_PATH = f"{PROJECT_FOLDER}/compliance_report.pdf"

# Build the PDF document
doc = SimpleDocTemplate(
    OUTPUT_PATH,
    pagesize=A4,
    rightMargin=20*mm,
    leftMargin=20*mm,
    topMargin=20*mm,
    bottomMargin=25*mm,
    title="Transaction Monitoring Compliance Report",
    author="AI Compliance Monitoring System"
)

# Build with footer on every page
doc.build(
    story,
    onFirstPage=add_page_footer,
    onLaterPages=add_page_footer
)

print("=" * 50)
print("Compliance Report Generated Successfully")
print("=" * 50)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Transactions reported: {len(results_df)}")
print(f"Critical: {critical_count} | High: {high_count} | "
      f"Medium: {medium_count} | Low: {low_count}")

Compliance Report Generated Successfully
Saved to: /content/drive/MyDrive/compliance_ai_project/compliance_report.pdf
Transactions reported: 77
Critical: 23 | High: 28 | Medium: 25 | Low: 1


## Stage 4 — Daily Triage Report
A concise, action-focused report for the compliance team's
daily review. Covers Critical risk transactions and missing
KYC cases only. Designed to be reviewed in under 10 minutes.

In [24]:
# -----------------------------------------------
# FILTER DATA FOR TRIAGE REPORT
# -----------------------------------------------
import pandas as pd
from datetime import datetime

# Load results if not already in memory
results_df = pd.read_csv(f"{PROJECT_FOLDER}/investigation_results.csv")

# Filter 1: Critical transactions only
critical_df = results_df[
    results_df["ai_risk_score"] == "Critical"
].copy()

# Filter 2: Missing KYC passport (any risk level)
missing_kyc_df = results_df[
    results_df["sender_passport_no"].isna()
].copy()

# Remove duplicates — if a critical transaction also has
# missing KYC, it appears in critical section only
missing_kyc_df = missing_kyc_df[
    missing_kyc_df["ai_risk_score"] != "Critical"
]

# Pre-calculate manual review cases for cover page stats
manual_df = results_df[
    results_df["ai_assessment"].str.contains(
        "timed out|failed|unavailable",
        case=False,
        na=False
    )
].copy()

print(f"Manual review cases: {len(manual_df)}")

REPORT_DATE = datetime.now().strftime("%d %B %Y")
REPORT_TIME = datetime.now().strftime("%H:%M:%S")

print(f"Critical transactions: {len(critical_df)}")
print(f"Missing KYC (non-critical): {len(missing_kyc_df)}")
print(f"Total triage items: {len(critical_df) + len(missing_kyc_df)}")

Manual review cases: 13
Critical transactions: 23
Missing KYC (non-critical): 4
Total triage items: 27


In [25]:
# -----------------------------------------------
# BUILD DAILY TRIAGE REPORT
# -----------------------------------------------
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, PageBreak, HRFlowable
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
import re

# ---- Colours & Styles ----
NAVY        = colors.HexColor('#1B2A4A')
DARK_GREY   = colors.HexColor('#4A4A4A')
LIGHT_GREY  = colors.HexColor('#F5F5F5')
MID_GREY    = colors.HexColor('#CCCCCC')
WHITE       = colors.white
CRITICAL_RED   = colors.HexColor('#C0392B')
HIGH_ORANGE    = colors.HexColor('#E67E22')
MISSING_PURPLE = colors.HexColor('#8E44AD')

styles = getSampleStyleSheet()

style_section = ParagraphStyle(
    'Section', parent=styles['Heading1'],
    fontSize=13, textColor=NAVY,
    spaceBefore=10, spaceAfter=5
)
style_body = ParagraphStyle(
    'Body', parent=styles['Normal'],
    fontSize=10, textColor=DARK_GREY,
    spaceAfter=5, leading=14
)
style_small = ParagraphStyle(
    'Small', parent=styles['Normal'],
    fontSize=9, textColor=DARK_GREY,
    leading=13, spaceAfter=3
)
style_disclaimer = ParagraphStyle(
    'Disclaimer', parent=styles['Normal'],
    fontSize=8, textColor=MID_GREY,
    leading=11
)


def add_triage_footer(canvas, doc):
    """Footer for every page of the triage report."""
    canvas.saveState()
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(MID_GREY)
    canvas.setStrokeColor(MID_GREY)
    canvas.line(20*mm, 15*mm, 190*mm, 15*mm)
    canvas.drawString(
        20*mm, 10*mm,
        "CONFIDENTIAL — Daily Triage Report — Internal Use Only"
    )
    canvas.drawRightString(
        190*mm, 10*mm,
        f"Page {doc.page} | {REPORT_DATE}"
    )
    canvas.restoreState()


def clean_text(text):
    """Strips markdown formatting from AI responses."""
    if pd.isna(text):
        return "Assessment unavailable."
    text = re.sub(r'#+\s*', '', text)
    text = re.sub(r'\*\*', '', text)
    text = re.sub(r'\*', '', text)
    text = re.sub(r'---+', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


story = []

# ==================================================
# COVER PAGE
# ==================================================

story.append(Spacer(1, 35*mm))

# Urgency banner
banner_data = [["ACTION REQUIRED — DAILY COMPLIANCE TRIAGE"]]
banner = Table(banner_data, colWidths=[170*mm])
banner.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,-1), CRITICAL_RED),
    ('TEXTCOLOR', (0,0), (-1,-1), WHITE),
    ('FONTNAME', (0,0), (-1,-1), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 11),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('TOPPADDING', (0,0), (-1,-1), 8),
    ('BOTTOMPADDING', (0,0), (-1,-1), 8),
]))
story.append(banner)
story.append(Spacer(1, 10*mm))

# Title
story.append(Paragraph(
    "DAILY TRIAGE REPORT",
    ParagraphStyle('Title', parent=styles['Title'],
                   fontSize=28, textColor=NAVY,
                   alignment=TA_CENTER, spaceAfter=4)
))
story.append(Paragraph(
    "Transaction Monitoring — Priority Actions",
    ParagraphStyle('Sub', parent=styles['Normal'],
                   fontSize=13, textColor=DARK_GREY,
                   alignment=TA_CENTER, spaceAfter=4)
))
story.append(Spacer(1, 8*mm))
story.append(HRFlowable(
    width="80%", thickness=2,
    color=NAVY, spaceAfter=8*mm
))

# Cover metadata
meta_data = [
    ["Report Date:",        REPORT_DATE],
    ["Generated At:",       REPORT_TIME],
    ["Classification:",     "Confidential"],
    ["Department:",         "Compliance & Financial Crime"],
    ["Items Requiring Action:", str(len(critical_df) + len(missing_kyc_df))],
    ["Critical Transactions:",  str(len(critical_df))],
    ["Missing KYC Cases:",      str(len(missing_kyc_df))],
    ["Manual Review Required:", str(len(manual_df))],
]
meta_table = Table(meta_data, colWidths=[65*mm, 95*mm])
meta_table.setStyle(TableStyle([
    ('FONTNAME',  (0,0), (-1,-1), 'Helvetica'),
    ('FONTSIZE',  (0,0), (-1,-1), 11),
    ('FONTNAME',  (0,0), (0,-1), 'Helvetica-Bold'),
    ('TEXTCOLOR', (0,0), (0,-1), NAVY),
    ('TEXTCOLOR', (1,0), (1,-1), DARK_GREY),
    ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ('TOPPADDING',    (0,0), (-1,-1), 5),
    ('LINEBELOW', (0,-1), (-1,-1), 0.5, MID_GREY),
]))
story.append(meta_table)

story.append(Spacer(1, 10*mm))

# Instructions box
instructions_data = [[
    Paragraph(
        "<b>HOW TO USE THIS REPORT</b><br/>"
        "This report contains only items requiring action today. "
        "Critical transactions should be reviewed and escalated before end of business. "
        "Missing KYC cases should be assigned to the onboarding team immediately. "
        "For the full investigation log, refer to the Monthly Compliance Report.",
        ParagraphStyle('Inst', parent=styles['Normal'],
                       fontSize=9, textColor=DARK_GREY, leading=13)
    )
]]
instructions = Table(instructions_data, colWidths=[170*mm])
instructions.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,-1), LIGHT_GREY),
    ('TOPPADDING',    (0,0), (-1,-1), 8),
    ('BOTTOMPADDING', (0,0), (-1,-1), 8),
    ('LEFTPADDING',   (0,0), (-1,-1), 10),
    ('RIGHTPADDING',  (0,0), (-1,-1), 10),
    ('BOX', (0,0), (-1,-1), 1, NAVY),
]))
story.append(instructions)

story.append(PageBreak())


# ==================================================
# SECTION 1: CRITICAL TRANSACTIONS
# ==================================================

story.append(Paragraph(
    f"1. Critical Transactions ({len(critical_df)})",
    style_section
))
story.append(HRFlowable(
    width="100%", thickness=1,
    color=CRITICAL_RED, spaceAfter=4
))
story.append(Paragraph(
    "The following transactions have been assessed as Critical risk and require "
    "immediate escalation to the Senior Compliance Officer. Where transactions "
    "are pending, consider placing a hold pending investigation.",
    style_body
))
story.append(Spacer(1, 4*mm))

if len(critical_df) == 0:
    story.append(Paragraph(
        "No Critical risk transactions identified today.",
        style_body
    ))
else:
    for i, row in critical_df.iterrows():

        # Transaction header
        header_data = [[
            f"  {row['transaction_id']}  —  "
            f"£{float(row['amount_gbp']):,.2f}  —  "
            f"{row['transaction_type']}",
            "  CRITICAL  "
        ]]
        header = Table(header_data, colWidths=[135*mm, 35*mm])
        header.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (0,0), NAVY),
            ('BACKGROUND', (1,0), (1,0), CRITICAL_RED),
            ('TEXTCOLOR',  (0,0), (-1,-1), WHITE),
            ('FONTNAME',   (0,0), (-1,-1), 'Helvetica-Bold'),
            ('FONTSIZE',   (0,0), (-1,-1), 9),
            ('ALIGN',      (0,0), (0,0), 'LEFT'),
            ('ALIGN',      (1,0), (1,0), 'CENTER'),
            ('TOPPADDING',    (0,0), (-1,-1), 5),
            ('BOTTOMPADDING', (0,0), (-1,-1), 5),
            ('LEFTPADDING',   (0,0), (0,0), 8),
        ]))
        story.append(header)

        # Key details — compact two column layout
        passport = (row['sender_passport_no']
                    if pd.notna(row['sender_passport_no'])
                    else "MISSING")

        details_data = [
            ["Date & Time",   str(row['date_time']),
             "Status",        row['status']],
            ["Sender",        row['sender_name'],
             "KYC Risk",      row['sender_kyc_risk']],
            ["Nationality",   row['sender_nationality'],
             "Passport",      passport],
            ["Receiver",      row['receiver_name'],
             "Receiver Acc",  row['receiver_account_no']],
        ]
        details = Table(details_data,
                        colWidths=[30*mm, 60*mm, 30*mm, 50*mm])
        details.setStyle(TableStyle([
            ('FONTNAME',  (0,0), (-1,-1), 'Helvetica'),
            ('FONTSIZE',  (0,0), (-1,-1), 8),
            ('FONTNAME',  (0,0), (0,-1), 'Helvetica-Bold'),
            ('FONTNAME',  (2,0), (2,-1), 'Helvetica-Bold'),
            ('TEXTCOLOR', (0,0), (0,-1), NAVY),
            ('TEXTCOLOR', (2,0), (2,-1), NAVY),
            ('ROWBACKGROUNDS', (0,0), (-1,-1), [WHITE, LIGHT_GREY]),
            ('GRID',      (0,0), (-1,-1), 0.3, MID_GREY),
            ('TOPPADDING',    (0,0), (-1,-1), 3),
            ('BOTTOMPADDING', (0,0), (-1,-1), 3),
            ('LEFTPADDING',   (0,0), (-1,-1), 5),
        ]))
        story.append(details)

        # AI recommended action only (concise for triage)
        clean = clean_text(row['ai_assessment'])

        # Extract just the recommended action line
        action_text = ""
        for line in clean.split("\n"):
            if "RECOMMENDED ACTION" in line.upper():
                # Get the text after this line
                idx = clean.split("\n").index(line)
                action_lines = clean.split("\n")[idx+1:idx+4]
                action_text = " ".join(
                    [l.strip() for l in action_lines if l.strip()]
                )
                break

        if not action_text:
            action_text = clean[:300] + "..." if len(clean) > 300 else clean

        story.append(Paragraph(
            f"<b>Recommended Action:</b> {action_text}",
            ParagraphStyle('Action', parent=style_small,
                           leftIndent=8, spaceBefore=3,
                           textColor=DARK_GREY)
        ))
        story.append(Spacer(1, 5*mm))

story.append(PageBreak())


# ==================================================
# SECTION 2: MISSING KYC
# ==================================================

story.append(Paragraph(
    f"2. Missing KYC Documentation ({len(missing_kyc_df)})",
    style_section
))
story.append(HRFlowable(
    width="100%", thickness=1,
    color=MISSING_PURPLE, spaceAfter=4
))
story.append(Paragraph(
    "The following cases have incomplete KYC documentation — specifically a "
    "missing passport number. These customers must be contacted within 10 "
    "business days. Transaction restrictions should be considered until "
    "documentation is received.",
    style_body
))
story.append(Spacer(1, 4*mm))

if len(missing_kyc_df) == 0:
    story.append(Paragraph(
        "No missing KYC cases identified today.",
        style_body
    ))
else:
    # Summary table for quick scanning
    kyc_table_data = [
        ["Transaction ID", "Sender Name", "Amount (GBP)",
         "KYC Risk", "Transaction Type", "Status"]
    ]
    for i, row in missing_kyc_df.iterrows():
        kyc_table_data.append([
            row['transaction_id'],
            row['sender_name'],
            f"£{float(row['amount_gbp']):,.2f}",
            row['sender_kyc_risk'],
            row['transaction_type'],
            row['status']
        ])

    kyc_table = Table(
        kyc_table_data,
        colWidths=[28*mm, 45*mm, 28*mm, 22*mm, 32*mm, 22*mm]
    )
    kyc_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), MISSING_PURPLE),
        ('TEXTCOLOR',  (0,0), (-1,0), WHITE),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTNAME',   (0,1), (-1,-1), 'Helvetica'),
        ('FONTSIZE',   (0,0), (-1,-1), 8),
        ('ALIGN',      (0,0), (-1,-1), 'CENTER'),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [WHITE, LIGHT_GREY]),
        ('GRID',       (0,0), (-1,-1), 0.3, MID_GREY),
        ('TOPPADDING',    (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ]))
    story.append(kyc_table)

# ==================================================
# SECTION 3: MANUAL REVIEW REQUIRED
# ==================================================

manual_df = results_df[
    results_df["ai_assessment"].str.contains(
        "timed out|failed|unavailable",
        case=False,
        na=False
    )
].copy()

story.append(Paragraph(
    f"3. Manual Review Required ({len(manual_df)})",
    style_section
))
story.append(HRFlowable(
    width="100%", thickness=1,
    color=HIGH_ORANGE, spaceAfter=4
))

if len(manual_df) == 0:
    story.append(Paragraph(
        "All flagged transactions were successfully assessed by the "
        "AI monitoring system today. No manual reviews outstanding.",
        style_body
    ))
else:
    story.append(Paragraph(
        f"{len(manual_df)} transaction(s) could not be automatically assessed "
        "due to a system timeout. These require manual review before end of "
        "business today and should be treated as high priority.",
        style_body
    ))
    story.append(Spacer(1, 3*mm))

    # Compact manual review table for triage
    manual_triage_data = [
        ["Transaction ID", "Sender Name", "Amount (GBP)",
         "KYC Risk", "Type", "Status"]
    ]
    for i, row in manual_df.iterrows():
        manual_triage_data.append([
            row['transaction_id'],
            row['sender_name'],
            f"£{float(row['amount_gbp']):,.2f}",
            row['sender_kyc_risk'],
            row['transaction_type'],
            row['status']
        ])

    manual_triage_table = Table(
        manual_triage_data,
        colWidths=[28*mm, 45*mm, 28*mm, 22*mm, 32*mm, 22*mm]
    )
    manual_triage_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), HIGH_ORANGE),
        ('TEXTCOLOR',  (0,0), (-1,0), WHITE),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTNAME',   (0,1), (-1,-1), 'Helvetica'),
        ('FONTSIZE',   (0,0), (-1,-1), 8),
        ('ALIGN',      (0,0), (-1,-1), 'CENTER'),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [WHITE, LIGHT_GREY]),
        ('GRID',       (0,0), (-1,-1), 0.3, MID_GREY),
        ('TOPPADDING',    (0,0), (-1,-1), 4),
        ('BOTTOMPADDING', (0,0), (-1,-1), 4),
    ]))
    story.append(manual_triage_table)
    story.append(Spacer(1, 3*mm))

    # Urgent action box
    urgent_data = [[
        Paragraph(
            "<b>URGENT:</b> The above transactions have not been AI assessed. "
            "Assign to a compliance analyst immediately for manual risk "
            "scoring and recommended action documentation.",
            ParagraphStyle('Urgent', parent=styles['Normal'],
                           fontSize=9, textColor=DARK_GREY, leading=13)
        )
    ]]
    urgent_box = Table(urgent_data, colWidths=[170*mm])
    urgent_box.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,-1), colors.HexColor('#FEF9E7')),
        ('TOPPADDING',    (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LEFTPADDING',   (0,0), (-1,-1), 10),
        ('RIGHTPADDING',  (0,0), (-1,-1), 10),
        ('BOX', (0,0), (-1,-1), 1, HIGH_ORANGE),
    ]))
    story.append(urgent_box)
    story.append(Spacer(1, 5*mm))

# ==================================================
# CLOSING NOTE
# ==================================================

story.append(Spacer(1, 10*mm))
story.append(HRFlowable(
    width="100%", thickness=0.5,
    color=MID_GREY, spaceAfter=4
))
story.append(Paragraph(
    "This triage report was generated automatically by the AI Transaction "
    "Monitoring System and covers priority items only. For the complete "
    "investigation log including Medium and Low risk transactions, refer to "
    "the full Compliance Report. All findings should be validated by a "
    "qualified compliance professional before action is taken.",
    style_disclaimer
))

print("Triage report content built successfully")
print(f"Critical items: {len(critical_df)}")
print(f"Missing KYC items: {len(missing_kyc_df)}")

Triage report content built successfully
Critical items: 23
Missing KYC items: 4


In [26]:
# -----------------------------------------------
# GENERATE TRIAGE PDF
# -----------------------------------------------

TRIAGE_PATH = f"{PROJECT_FOLDER}/daily_triage_report.pdf"

doc = SimpleDocTemplate(
    TRIAGE_PATH,
    pagesize=A4,
    rightMargin=20*mm,
    leftMargin=20*mm,
    topMargin=20*mm,
    bottomMargin=25*mm,
    title="Daily Compliance Triage Report",
    author="AI Compliance Monitoring System"
)

doc.build(
    story,
    onFirstPage=add_triage_footer,
    onLaterPages=add_triage_footer
)

print("=" * 50)
print("Daily Triage Report Generated Successfully")
print("=" * 50)
print(f"Saved to: {TRIAGE_PATH}")
print(f"Total action items: {len(critical_df) + len(missing_kyc_df)}")

Daily Triage Report Generated Successfully
Saved to: /content/drive/MyDrive/compliance_ai_project/daily_triage_report.pdf
Total action items: 27
